# 05 Model Comparison and Error Analysis

This notebook compares the **TF-IDF + Logistic Regression** baseline against the **BioBERT** transformer model using the saved output files.


## Purpose

Use this notebook after training to answer the key research questions:

- Did BioBERT improve over the classical baseline?
- On which kinds of examples does BioBERT help?
- Where do both models still fail?
- What examples are useful for qualitative discussion in the thesis/report?


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report


In [2]:
PROJECT_ROOT = Path(r'C:\Users\ribam\Desktop\Reseach\Dataset')
OUTPUT_ROOT = PROJECT_ROOT / 'output'
COMPARE_DIR = OUTPUT_ROOT / 'model_comparison'
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

TFIDF_METRICS_PATH = OUTPUT_ROOT / 'tfidf_logreg_metrics.json'
TFIDF_PRED_PATH = OUTPUT_ROOT / 'tfidf_logreg_test_predictions.csv'

BIOBERT_DIR = OUTPUT_ROOT / 'biobert_fakehealth_healthfact'
BIOBERT_METRICS_PATH = BIOBERT_DIR / 'biobert_metrics.json'
BIOBERT_PRED_PATH = BIOBERT_DIR / 'biobert_test_predictions.csv'

TFIDF_METRICS_PATH, BIOBERT_METRICS_PATH


(WindowsPath('C:/Users/ribam/Desktop/Reseach/Dataset/output/tfidf_logreg_metrics.json'),
 WindowsPath('C:/Users/ribam/Desktop/Reseach/Dataset/output/biobert_fakehealth_healthfact/biobert_metrics.json'))

In [3]:
with open(TFIDF_METRICS_PATH, 'r', encoding='utf-8') as f:
    tfidf_metrics = json.load(f)

with open(BIOBERT_METRICS_PATH, 'r', encoding='utf-8') as f:
    biobert_metrics = json.load(f)

tfidf_metrics, biobert_metrics


({'model': 'TF-IDF + Logistic Regression',
  'text_column': 'model_text',
  'train_rows': 9708,
  'dev_rows': 1338,
  'test_rows': 1317,
  'curated_train_repeat': 4,
  'dev_metrics': {'split': 'dev',
   'accuracy': 0.7227204783258595,
   'precision': 0.7808564231738035,
   'recall': 0.758873929008568,
   'f1': 0.7697082557417753,
   'macro_precision': 0.7093620351163135,
   'macro_recall': 0.7124504002048597,
   'macro_f1': 0.710675724115019,
   'misinformation_precision': 0.6378676470588235,
   'misinformation_recall': 0.6660268714011516,
   'misinformation_f1': 0.6516431924882629,
   'reliable_precision': 0.7808564231738035,
   'reliable_recall': 0.758873929008568,
   'reliable_f1': 0.7697082557417753},
  'test_metrics': {'split': 'test',
   'accuracy': 0.7205770690964313,
   'precision': 0.7932960893854749,
   'recall': 0.7208121827411168,
   'f1': 0.7553191489361702,
   'macro_precision': 0.7136197585030536,
   'macro_recall': 0.720519512920653,
   'macro_f1': 0.7148277160610055,
 

In [4]:
comparison_df = pd.DataFrame([
    {
        'model': 'TF-IDF + Logistic Regression',
        'dev_accuracy': tfidf_metrics['dev_metrics']['accuracy'],
        'dev_precision': tfidf_metrics['dev_metrics']['precision'],
        'dev_recall': tfidf_metrics['dev_metrics']['recall'],
        'dev_f1': tfidf_metrics['dev_metrics']['f1'],
        'test_accuracy': tfidf_metrics['test_metrics']['accuracy'],
        'test_precision': tfidf_metrics['test_metrics']['precision'],
        'test_recall': tfidf_metrics['test_metrics']['recall'],
        'test_f1': tfidf_metrics['test_metrics']['f1'],
    },
    {
        'model': 'BioBERT',
        'dev_accuracy': biobert_metrics['dev_metrics']['eval_accuracy'],
        'dev_precision': biobert_metrics['dev_metrics']['eval_precision'],
        'dev_recall': biobert_metrics['dev_metrics']['eval_recall'],
        'dev_f1': biobert_metrics['dev_metrics']['eval_f1'],
        'test_accuracy': biobert_metrics['test_metrics']['eval_accuracy'],
        'test_precision': biobert_metrics['test_metrics']['eval_precision'],
        'test_recall': biobert_metrics['test_metrics']['eval_recall'],
        'test_f1': biobert_metrics['test_metrics']['eval_f1'],
    },
])

comparison_df.round(4)


,model,dev_accuracy,dev_precision,dev_recall,dev_f1,test_accuracy,test_precision,test_recall,test_f1
0,TF-IDF + Logistic Regression,0.7227,0.7809,0.7589,0.7697,0.7206,0.7933,0.7208,0.7553
1,BioBERT,0.7638,0.7930,0.8299,0.8110,0.7563,0.7945,0.7995,0.7970


In [5]:
comparison_df.to_csv(COMPARE_DIR / 'model_metric_comparison.csv', index=False)
print('Saved metric comparison to', COMPARE_DIR / 'model_metric_comparison.csv')


Saved metric comparison to C:\Users\ribam\Desktop\Reseach\Dataset\output\model_comparison\model_metric_comparison.csv


In [6]:
tfidf_pred = pd.read_csv(TFIDF_PRED_PATH)
biobert_pred = pd.read_csv(BIOBERT_PRED_PATH)

tfidf_pred = tfidf_pred.rename(columns={
    "prediction": "tfidf_prediction",
    "prob_class_0": "tfidf_prob_0",
    "prob_class_1": "tfidf_prob_1",
})

biobert_pred = biobert_pred.rename(columns={
    "prediction": "biobert_prediction",
    "prob_class_0": "biobert_prob_0",
    "prob_class_1": "biobert_prob_1",
})

JOIN_KEYS = ["dataset", "split", "record_id", "label"]


def existing_columns(frame, columns):
    return [column for column in columns if column in frame.columns]


tfidf_cols = existing_columns(
    tfidf_pred,
    JOIN_KEYS + ["text", "model_text", "tfidf_prediction", "tfidf_prob_0", "tfidf_prob_1"],
)
biobert_cols = existing_columns(
    biobert_pred,
    JOIN_KEYS + ["biobert_prediction", "biobert_prob_0", "biobert_prob_1"],
)

merged = tfidf_pred[tfidf_cols].merge(
    biobert_pred[biobert_cols],
    on=JOIN_KEYS,
    how="inner",
)

if "model_text" not in merged.columns:
    merged["model_text"] = merged["text"]

merged["tfidf_confidence"] = np.where(
    merged["tfidf_prediction"] == 1,
    merged.get("tfidf_prob_1", np.nan),
    merged.get("tfidf_prob_0", np.nan),
)
merged["biobert_confidence"] = np.where(
    merged["biobert_prediction"] == 1,
    merged.get("biobert_prob_1", np.nan),
    merged.get("biobert_prob_0", np.nan),
)

print("Merged rows:", len(merged))
print("Merged columns:", list(merged.columns))
merged.head(3)


Merged rows: 1317
Merged columns: ['dataset', 'split', 'record_id', 'label', 'text', 'model_text', 'tfidf_prediction', 'tfidf_prob_0', 'tfidf_prob_1', 'biobert_prediction', 'biobert_prob_0', 'biobert_prob_1', 'tfidf_confidence', 'biobert_confidence']


,dataset,split,record_id,label,text,model_text,tfidf_prediction,tfidf_prob_0,tfidf_prob_1,biobert_prediction,biobert_prob_0,biobert_prob_1,tfidf_confidence,biobert_confidence
0,healthfact,test,33456,0,A mother revealed to her child in a letter aft...,A mother revealed to her child in a letter aft...,0,0.866750,0.133250,0,0.693878,0.306122,0.866750,0.693878
1,healthfact,test,2542,1,Study says too many Americans still drink too ...,Study says too many Americans still drink too ...,1,0.143167,0.856833,1,0.007271,0.992730,0.856833,0.992730
2,healthfact,test,26678,1,Viral image Says 80% of novel coronavirus case...,Viral image Says 80% of novel coronavirus case...,0,0.716531,0.283469,1,0.005124,0.994876,0.716531,0.994876


In [7]:
merged['tfidf_correct'] = merged['tfidf_prediction'] == merged['label']
merged['biobert_correct'] = merged['biobert_prediction'] == merged['label']
merged['models_agree'] = merged['tfidf_prediction'] == merged['biobert_prediction']

merged['comparison_bucket'] = 'both_wrong'
merged.loc[merged['tfidf_correct'] & merged['biobert_correct'], 'comparison_bucket'] = 'both_correct'
merged.loc[merged['tfidf_correct'] & ~merged['biobert_correct'], 'comparison_bucket'] = 'tfidf_only_correct'
merged.loc[~merged['tfidf_correct'] & merged['biobert_correct'], 'comparison_bucket'] = 'biobert_only_correct'

merged['comparison_bucket'].value_counts()


comparison_bucket
both_correct            825
both_wrong              197
biobert_only_correct    171
tfidf_only_correct      124
Name: count, dtype: int64

In [8]:
bucket_summary = merged['comparison_bucket'].value_counts().rename_axis('bucket').reset_index(name='count')
bucket_summary['percent'] = (bucket_summary['count'] / len(merged) * 100).round(2)
bucket_summary


,bucket,count,percent
0,both_correct,825,62.64
1,both_wrong,197,14.96
2,biobert_only_correct,171,12.98
3,tfidf_only_correct,124,9.42


In [9]:
bucket_summary.to_csv(COMPARE_DIR / 'comparison_bucket_summary.csv', index=False)
merged.to_csv(COMPARE_DIR / 'merged_model_predictions.csv', index=False)
print('Saved merged comparison files to', COMPARE_DIR)


Saved merged comparison files to C:\Users\ribam\Desktop\Reseach\Dataset\output\model_comparison


In [10]:
dataset_bucket = pd.crosstab(merged['dataset'], merged['comparison_bucket'])
label_bucket = pd.crosstab(merged['label'], merged['comparison_bucket'])

display(dataset_bucket)
display(label_bucket)


comparison_bucket,biobert_only_correct,both_correct,both_wrong,tfidf_only_correct
dataset,,,,
curated_health_claims,1,4,1,0
fakehealth,64,135,75,50
healthfact,106,686,121,74


comparison_bucket,biobert_only_correct,both_correct,both_wrong,tfidf_only_correct
label,,,,
0,59,307,89,74
1,112,518,108,50


In [11]:
dataset_bucket.to_csv(COMPARE_DIR / 'dataset_bucket_crosstab.csv')
label_bucket.to_csv(COMPARE_DIR / 'label_bucket_crosstab.csv')


## Qualitative Error Analysis

These tables are especially useful for the research write-up because they give concrete examples of where the transformer helps or still fails.


In [12]:
analysis_columns = [
    "dataset",
    "record_id",
    "label",
    "tfidf_prediction",
    "biobert_prediction",
    "tfidf_prob_0",
    "tfidf_prob_1",
    "tfidf_confidence",
    "biobert_prob_0",
    "biobert_prob_1",
    "biobert_confidence",
    "text",
    "model_text",
]
analysis_columns = [column for column in analysis_columns if column in merged.columns]

biobert_wins = merged.loc[
    merged["comparison_bucket"] == "biobert_only_correct",
    analysis_columns,
].copy()

tfidf_wins = merged.loc[
    merged["comparison_bucket"] == "tfidf_only_correct",
    analysis_columns,
].copy()

both_wrong = merged.loc[
    merged["comparison_bucket"] == "both_wrong",
    analysis_columns,
].copy()

both_correct = merged.loc[
    merged["comparison_bucket"] == "both_correct",
    analysis_columns,
].copy()

print("BioBERT only correct:", len(biobert_wins))
print("TF-IDF only correct:", len(tfidf_wins))
print("Both wrong:", len(both_wrong))
print("Both correct:", len(both_correct))


BioBERT only correct: 171
TF-IDF only correct: 124
Both wrong: 197
Both correct: 825


In [13]:
biobert_wins.head(10)


,dataset,record_id,label,tfidf_prediction,biobert_prediction,tfidf_prob_0,tfidf_prob_1,tfidf_confidence,biobert_prob_0,biobert_prob_1,biobert_confidence,text,model_text
2,healthfact,26678,1,0,1,0.716531,0.283469,0.716531,0.005124,0.994876,0.994876,Viral image Says 80% of novel coronavirus case...,Viral image Says 80% of novel coronavirus case...
15,healthfact,29144,0,1,0,0.478214,0.521786,0.521786,0.530698,0.469302,0.530698,Queen Elizabeth II wore a Burmese Ruby Tiara a...,Queen Elizabeth II wore a Burmese Ruby Tiara a...
18,healthfact,9960,0,1,0,0.389336,0.610664,0.610664,0.692555,0.307445,0.692555,Scientists look to stem cells to mend broken h...,Scientists look to stem cells to mend broken h...
19,healthfact,11523,0,1,0,0.283820,0.716180,0.716180,0.510304,0.489696,0.510304,Study suggests a second LDL test,Study suggests a second LDL test
60,healthfact,10970,1,0,1,0.507355,0.492645,0.507355,0.117939,0.882062,0.882062,"To reverse damage of sitting, take a brisk, ho...","To reverse damage of sitting, take a brisk, ho..."
66,healthfact,22800,1,0,1,0.500691,0.499309,0.500691,0.278809,0.721191,0.721191,The city of Atlanta has $56 million in its res...,The city of Atlanta has $56 million in its res...
67,healthfact,5571,1,0,1,0.742110,0.257890,0.742110,0.008738,0.991262,0.991262,CVI: The impairment affecting children whose e...,CVI: The impairment affecting children whose e...
69,healthfact,28092,1,0,1,0.732832,0.267168,0.732832,0.417868,0.582132,0.582132,Immigration authorities are taking rosaries aw...,Immigration authorities are taking rosaries aw...
72,healthfact,33919,0,1,0,0.499879,0.500121,0.500121,0.609493,0.390507,0.609493,A list compiles Andy Rooney's wry observations...,A list compiles Andy Rooney's wry observations...
82,healthfact,33983,0,1,0,0.449918,0.550082,0.550082,0.822461,0.177540,0.822461,NASA and NOAA faked climate data in the GISTEM...,NASA and NOAA faked climate data in the GISTEM...


In [14]:
tfidf_wins.head(10)


,dataset,record_id,label,tfidf_prediction,biobert_prediction,tfidf_prob_0,tfidf_prob_1,tfidf_confidence,biobert_prob_0,biobert_prob_1,biobert_confidence,text,model_text
24,healthfact,16513,0,0,1,0.853234,0.146766,0.853234,0.088042,0.911958,0.911958,"Obamacare ""cuts seniors’ Medicare.","Obamacare ""cuts seniors’ Medicare."
28,healthfact,28214,1,1,0,0.399102,0.600898,0.600898,0.838921,0.161079,0.838921,"People can fly with dogs, pot-bellied pigs, tu...","People can fly with dogs, pot-bellied pigs, tu..."
50,healthfact,9255,1,1,0,0.240797,0.759203,0.759203,0.741116,0.258884,0.741116,FDA Approves Bayer's Kyleena™ (Levonorgestrel-...,FDA Approves Bayer's Kyleena™ (Levonorgestrel-...
58,healthfact,9614,1,1,0,0.273807,0.726193,0.726193,0.616215,0.383785,0.616215,Expert Panel Reaffirms Need for Colon Cancer S...,Expert Panel Reaffirms Need for Colon Cancer S...
59,healthfact,6881,1,1,0,0.329880,0.670120,0.670120,0.942362,0.057638,0.942362,Okposo says he’s healthy after concussion put ...,Okposo says he’s healthy after concussion put ...
75,healthfact,36264,0,0,1,0.854676,0.145324,0.854676,0.493516,0.506484,0.506484,"An image shows reported cases of ""flesh eating...","An image shows reported cases of ""flesh eating..."
76,healthfact,35389,1,1,0,0.230295,0.769705,0.769705,0.509099,0.490901,0.509099,Some grocery store receipts contain chemicals ...,Some grocery store receipts contain chemicals ...
84,healthfact,13872,1,1,0,0.258910,0.741090,0.741090,0.582096,0.417904,0.582096,"Illegal tobacco sales, price driven too high, ...","Illegal tobacco sales, price driven too high, ..."
91,healthfact,37950,1,1,0,0.445895,0.554105,0.554105,0.660838,0.339162,0.660838,"On September 18 2020, Twitter user @JohnCammo ...","On September 18 2020, Twitter user @JohnCammo ..."
109,healthfact,9395,0,0,1,0.596819,0.403181,0.596819,0.405719,0.594281,0.594281,Menopause 'hot flash' medicine could cut sympt...,Menopause 'hot flash' medicine could cut sympt...


In [15]:
both_wrong.head(10)


,dataset,record_id,label,tfidf_prediction,biobert_prediction,tfidf_prob_0,tfidf_prob_1,tfidf_confidence,biobert_prob_0,biobert_prob_1,biobert_confidence,text,model_text
23,healthfact,10667,0,1,1,0.436146,0.563854,0.563854,0.410598,0.589402,0.589402,Artificial Pancreas Continues to Show Promise,Artificial Pancreas Continues to Show Promise
41,healthfact,13528,1,0,0,0.635908,0.364092,0.635908,0.576324,0.423676,0.576324,"Heroin comes in the United States ""from the so...","Heroin comes in the United States ""from the so..."
47,healthfact,8959,0,1,1,0.497350,0.502650,0.502650,0.070198,0.929802,0.929802,Too many people missing out on health benefits...,Too many people missing out on health benefits...
52,healthfact,16366,1,0,0,0.780730,0.219270,0.780730,0.550063,0.449937,0.550063,"Charlie Crist Says Rick Scott signed ""laws req...","Charlie Crist Says Rick Scott signed ""laws req..."
68,healthfact,16390,0,1,1,0.490831,0.509169,0.509169,0.489311,0.510689,0.510689,"Leticia Van de Putte ""voted to give illegal im...","Leticia Van de Putte ""voted to give illegal im..."
89,healthfact,27567,1,0,0,0.730714,0.269286,0.730714,0.614400,0.385600,0.614400,The FDA has updated its warnings for — and rec...,The FDA has updated its warnings for — and rec...
97,healthfact,10507,1,0,0,0.584157,0.415843,0.584157,0.646674,0.353326,0.646674,The Healthy Skeptic: Products make testosteron...,The Healthy Skeptic: Products make testosteron...
100,healthfact,9673,0,1,1,0.196998,0.803002,0.803002,0.073789,0.926211,0.926211,'Female Viagra' gets mixed reviews,'Female Viagra' gets mixed reviews
106,healthfact,16846,1,0,0,0.659197,0.340803,0.659197,0.832043,0.167957,0.832043,After Massachusetts passed a mandatory health ...,After Massachusetts passed a mandatory health ...
107,healthfact,10748,1,0,0,0.582473,0.417527,0.582473,0.937458,0.062542,0.937458,"Humira Provides Effective, Non-Steroid Alterna...","Humira Provides Effective, Non-Steroid Alterna..."


In [16]:
biobert_wins.to_csv(COMPARE_DIR / "biobert_only_correct_examples.csv", index=False)
tfidf_wins.to_csv(COMPARE_DIR / "tfidf_only_correct_examples.csv", index=False)
both_wrong.to_csv(COMPARE_DIR / "both_wrong_examples.csv", index=False)
both_correct.to_csv(COMPARE_DIR / "both_correct_examples.csv", index=False)
print("Saved example CSVs for qualitative analysis.")


Saved example CSVs for qualitative analysis.


In [17]:
summary = {
    "best_model_by_test_f1": comparison_df.sort_values("test_f1", ascending=False).iloc[0]["model"],
    "tfidf_test_accuracy": float(comparison_df.loc[comparison_df["model"] == "TF-IDF + Logistic Regression", "test_accuracy"].iloc[0]),
    "biobert_test_accuracy": float(comparison_df.loc[comparison_df["model"] == "BioBERT", "test_accuracy"].iloc[0]),
    "tfidf_test_f1": float(comparison_df.loc[comparison_df["model"] == "TF-IDF + Logistic Regression", "test_f1"].iloc[0]),
    "biobert_test_f1": float(comparison_df.loc[comparison_df["model"] == "BioBERT", "test_f1"].iloc[0]),
    "f1_gain_biobert_over_tfidf": float(
        comparison_df.loc[comparison_df["model"] == "BioBERT", "test_f1"].iloc[0]
        - comparison_df.loc[comparison_df["model"] == "TF-IDF + Logistic Regression", "test_f1"].iloc[0]
    ),
    "bucket_counts": bucket_summary.set_index("bucket")["count"].to_dict(),
    "qualitative_csvs": {
        "biobert_only_correct": str(COMPARE_DIR / "biobert_only_correct_examples.csv"),
        "tfidf_only_correct": str(COMPARE_DIR / "tfidf_only_correct_examples.csv"),
        "both_wrong": str(COMPARE_DIR / "both_wrong_examples.csv"),
        "both_correct": str(COMPARE_DIR / "both_correct_examples.csv"),
    },
}

(COMPARE_DIR / "comparison_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
summary


{'best_model_by_test_f1': 'BioBERT',
 'tfidf_test_accuracy': 0.7205770690964313,
 'biobert_test_accuracy': 0.7562642369020501,
 'tfidf_test_f1': 0.7553191489361702,
 'biobert_test_f1': 0.7969639468690702,
 'f1_gain_biobert_over_tfidf': 0.04164479793289999,
 'bucket_counts': {'both_correct': 825,
  'both_wrong': 197,
  'biobert_only_correct': 171,
  'tfidf_only_correct': 124},
 'qualitative_csvs': {'biobert_only_correct': 'C:\\Users\\ribam\\Desktop\\Reseach\\Dataset\\output\\model_comparison\\biobert_only_correct_examples.csv',
  'tfidf_only_correct': 'C:\\Users\\ribam\\Desktop\\Reseach\\Dataset\\output\\model_comparison\\tfidf_only_correct_examples.csv',
  'both_wrong': 'C:\\Users\\ribam\\Desktop\\Reseach\\Dataset\\output\\model_comparison\\both_wrong_examples.csv',
  'both_correct': 'C:\\Users\\ribam\\Desktop\\Reseach\\Dataset\\output\\model_comparison\\both_correct_examples.csv'}}

## Next Step

After this notebook, the strongest next extension is either:

- **PubMedBERT** as a second domain-specific transformer comparison, or
- a **single-claim inference notebook** for demo/testing with custom user input.
